# Exp D - TabPFN v2 na Colab T4 GPU-u
Potrebno: Runtime > Change runtime type > **T4 GPU**. Ćelije izvršavaj redom. Težine: samo lokalni TabPFN v2 s Hugging Facea, hosted API se ne koristi.

## 1. Provjera GPU-a

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'nvidia-smi: nema izlaza')
import torch
if not torch.cuda.is_available():
    raise SystemExit('NEMA GPU-a. Runtime > Change runtime type > T4 GPU, pa ponovno pokreni od vrha.')
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Upload zipa i raspakiranje
Upload `exp_d_colab.zip` (skripte + 3 anonimizirana fajla).

In [ ]:
import os, zipfile
from google.colab import files
ROOT = '/content/expd'
os.makedirs(ROOT, exist_ok=True)
up = files.upload()
name = next(n for n in up if n.endswith('.zip'))
with zipfile.ZipFile(name) as z:
    z.extractall(ROOT)
os.makedirs(f'{ROOT}/out', exist_ok=True)
for r, _, fs in os.walk(ROOT):
    for f in fs:
        print(os.path.join(r, f), os.path.getsize(os.path.join(r, f)))

## 3. Instalacija (tabpfn 2.1.3)
Colab torch/numpy/pandas ostaju kakvi jesu; instalira se samo tabpfn s ovisnostima.

In [ ]:
!pip install -q tabpfn==2.1.3 einops huggingface-hub
import importlib.metadata as md
for p in ['tabpfn', 'torch', 'numpy', 'pandas', 'scikit-learn', 'pyarrow']:
    print(p, md.version(p))

## 4. Težine s Hugging Facea + SHA256
Prior-Labs/TabPFN-v2-clf. Ako hash ne odgovara, fajl se briše i stajemo.

In [ ]:
import hashlib, os
from huggingface_hub import hf_hub_download
EXPECTED = 'f65a35685aeef42e31b796d9bfa34e68d6fc780bc98e7bff7763802964cf435f'
ckpt = hf_hub_download('Prior-Labs/TabPFN-v2-clf', 'tabpfn-v2-classifier.ckpt', local_dir=f'{ROOT}/weights')
h = hashlib.sha256()
with open(ckpt, 'rb') as f:
    for b in iter(lambda: f.read(1 << 20), b''):
        h.update(b)
print(ckpt, os.path.getsize(ckpt), h.hexdigest())
if h.hexdigest() != EXPECTED:
    os.remove(ckpt)
    raise SystemExit('SHA256 NE ODGOVARA - težine obrisane, ne nastavljaj.')
os.environ['TABPFN_V2_CKPT'] = ckpt
os.environ.pop('TABPFN_TOKEN', None)
print('SHA256 OK')

## 5. Provjera podataka (dry-run, bez modela)

In [ ]:
!cd {ROOT} && python scripts/run_walkforward_tabpfn.py --workdir {ROOT}/ws --folds 1,2,3,4 --dry-run

## 6. TIMING - kontekst 10k redova
**Cilj: ispod 30 min po pokretanju.** Dalje (ćelija 7) se ide tek ako je ovo u redu. Procjena ispod je za 10k konteksta + 7.214 test redaka; stvarno pokretanje dodaje i kalibracijski isječak (~4-5k redaka).

In [ ]:
!cd {ROOT} && python scripts/tabpfn_smoke_timing.py --contexts 10000 --n-test 2000 --device cuda

## 7. GLAVNI eksperiment (checkpoint / resume)
Svako pokretanje sprema u `out/<run_id>/` i preskače se ako već postoji (`DONE`). Nakon prekida samo ponovno pokreni ćelije 1-4 (i ovu; podaci ostaju u /content/expd ako se sesija nije resetirala, inače ponovi upload).
Glavni split = fold 4 (18 pokretanja). Walk-forward = foldovi 1-3, sa `WF_CONTEXT` i seedom 42 (fold 4 je već gore).

In [ ]:
import itertools, json, os, subprocess, time, sys
TIMING_OK = False   # postavi True tek kad je timing iz ćelije 6 prihvatljiv (< 30 min)
WF_CONTEXT = 'last' # kontekst za walk-forward; po želji 'random' nakon pregleda glavnog splita
RUN_WALKFORWARD = True
if not TIMING_OK:
    raise SystemExit('Prvo pogledaj timing (ćelija 6) pa postavi TIMING_OK = True.')

VARIANTS = ['D-Aprime', 'D-Bprime', 'D-Bprime-par']
runs = [(4, v, m, s) for v, m, s in itertools.product(VARIANTS, ['last', 'random'], [42, 1, 2])]
if RUN_WALKFORWARD:
    runs += [(k, v, WF_CONTEXT, 42) for k in (1, 2, 3) for v in VARIANTS]

def run_id(k, v, m, s):
    return f'f{k}_{v}__{m}10k__s{s}'

t_all = time.time()
for i, (k, v, m, s) in enumerate(runs, 1):
    rid = run_id(k, v, m, s)
    od = f'{ROOT}/out/{rid}'
    if os.path.exists(f'{od}/DONE'):
        print(f'[{i}/{len(runs)}] skip {rid} (gotovo)')
        continue
    os.makedirs(od, exist_ok=True)
    print(f'[{i}/{len(runs)}] {rid}', flush=True)
    t0 = time.time()
    cmd = [sys.executable, 'scripts/run_walkforward_tabpfn.py', '--workdir', f'{ROOT}/ws', '--variants', v,
           '--contexts', m, '--seeds', str(s), '--folds', str(k), '--device', 'cuda', '--out-dir', od]
    p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'{od}/log.txt', 'w', encoding='utf-8') as lg:
        for line in p.stdout:
            print(line, end=''); lg.write(line)
    if p.wait() != 0:
        raise SystemExit(f'Pokretanje {rid} nije uspjelo (vidi ispis iznad). Ponovno pokretanje ćelije nastavlja od ovdje.')
    json.dump({'run_id': rid, 'minutes': round((time.time() - t0) / 60, 1)}, open(f'{od}/DONE', 'w'))
    print(f'    {rid}: {(time.time() - t0) / 60:.1f} min (ukupno {(time.time() - t_all) / 60:.0f} min)', flush=True)
print('SVE GOTOVO')

## 8. Zipanje rezultata i preuzimanje

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/exp_d_rezultati', 'zip', f'{ROOT}/out')
print(os.path.getsize('/content/exp_d_rezultati.zip'), 'B')
files.download('/content/exp_d_rezultati.zip')